In [ ]:
#pip uninstall chromadb

In [ ]:
#!pip install --upgrade chromadb --user

---
#### Overview of ChromaDB
---

ChromaDB is an open-source vector database designed for AI and machine learning applications, particularly for `embedding-based retrieval in RAG (Retrieval-Augmented Generation)`, semantic search, and recommendation systems.

#### Key Features

- Vector Store for AI Applications:
    - Stores and retrieves high-dimensional embeddings for text, images, etc.
    - Supports semantic similarity search.

- Python-first Experience:
    - Lightweight and easy-to-use Python API.
    - Ideal for quick prototyping to production.

- Open-source & Lightweight:
    - Pure Python, no heavy dependencies.
    - Runs locally or in a Docker container.

- Embedding Management:
    - Attach `metadata` and `document text` with embeddings.
    - Retrieve top-k results efficiently.

- Fast Similarity Search:
    - Uses `cosine similarity`, `L2 distance`, or `dot product`.
    - Scales to millions of embeddings.

- Persistence:
    - Supports in-memory or on-disk storage (SQLite backend).

- Integration-friendly:
    - Works with OpenAI, HuggingFace, Cohere embeddings.
    - Compatible with LangChain, LlamaIndex, FastAPI, Haystack, etc.

#### Hosting / Deployment Options

1. Local / Embedded Mode:
    - Runs inside your Python process.
    - Best for experiments or small-scale apps.

2. Persistent Local Server:
    - Use the `chromadb` CLI to start a server with data persistence.
    - Example: `chromadb run --path /data/chroma`

3. Dockerized Deployment:
    - Run ChromaDB as a container:
      docker run -p 8000:8000 chromadb/chroma

4. Cloud / Self-managed Hosting:
    - Deploy on AWS, GCP, Azure using Docker or Kubernetes.
    - Suitable for multi-user or production-scale RAG applications.

#### Other Key Points

- Scalability:
    - Handles `millions of embeddings` efficiently.
    - For `billion-scale`, consider distributed solutions like `Pinecone` or `Weaviate`.

- Simple Schema:
    - Data is organized in collections with:
        {id, `embedding`, `metadata`, `document`}

- Open-source License:
    - MIT License (free to use and self-host).

In [2]:
import chromadb
from chromadb.config import Settings

#### Example 01

In the ChromaDB client, the `allow_reset` parameter allows the user to reset 
the in-memory state or database during the client's lifecycle.

When `allow_reset = True`, the following behaviors are enabled:
 - It allows clearing all stored embeddings and metadata, effectively resetting ChromaDB.
 - Useful for testing, temporary data handling, or freeing up resources.
 - In production, it is recommended to set this to `False` to avoid unintended data loss.

In [3]:
# Create a ChromaDB client (in-memory for demo)
chroma_client = chromadb.Client(Settings(allow_reset=True))

In [4]:
# Create a collection
collection = chroma_client.create_collection(name="my_collection")

In [5]:
# C → Create (Insert Data)

# Add documents with IDs, metadata, and embeddings
collection.add(
    ids       = ["doc1", "doc2"],
    documents = ["Machine learning is amazing.", "Deep learning powers AI."],
    metadatas = [{"topic": "ml"}, {"topic": "ai"}],
    embeddings= [[0.1, 0.2, 0.3], [0.4, 0.5, 0.6]]
)

print("Data added successfully")

Data added successfully


providing ids in ChromaDB is optional.

If you don’t specify ids, ChromaDB will automatically generate unique UUIDs for each added document. 

However, it is recommended to provide IDs for easier management of updates and deletions later.

In [6]:
import uuid

docs = ["AI is transforming industries.", 
        "Vector databases are essential for RAG."]

# Adding documents without IDs
collection.add(
    ids        = [str(uuid.uuid4()) for _ in docs],
    documents  = docs,
    embeddings = [[0.1, 0.2, 0.3], [0.4, 0.5, 0.6]]
)

In [7]:
# Fetching all data
collection.get()

{'ids': ['doc1',
  'doc2',
  '63445d25-28d0-4813-b5de-b288988a1356',
  '3cbe0c7c-eb27-4050-b219-bfa47d1408e7'],
 'embeddings': None,
 'documents': ['Machine learning is amazing.',
  'Deep learning powers AI.',
  'AI is transforming industries.',
  'Vector databases are essential for RAG.'],
 'uris': None,
 'data': None,
 'metadatas': [{'topic': 'ml'}, {'topic': 'ai'}, None, None],
 'included': [<IncludeEnum.documents: 'documents'>,
  <IncludeEnum.metadatas: 'metadatas'>]}

Fetch Embeddings Explicitly

In [8]:
results = collection.get(include=['embeddings', 'documents', 'metadatas'])
print(results)

{'ids': ['doc1', 'doc2', '63445d25-28d0-4813-b5de-b288988a1356', '3cbe0c7c-eb27-4050-b219-bfa47d1408e7'], 'embeddings': array([[0.1       , 0.2       , 0.30000001],
       [0.40000001, 0.5       , 0.60000002],
       [0.1       , 0.2       , 0.30000001],
       [0.40000001, 0.5       , 0.60000002]]), 'documents': ['Machine learning is amazing.', 'Deep learning powers AI.', 'AI is transforming industries.', 'Vector databases are essential for RAG.'], 'uris': None, 'data': None, 'metadatas': [{'topic': 'ml'}, {'topic': 'ai'}, None, None], 'included': [<IncludeEnum.embeddings: 'embeddings'>, <IncludeEnum.documents: 'documents'>, <IncludeEnum.metadatas: 'metadatas'>]}


**reading data**

In [9]:
# Fetch all (without embeddings by default)
results_default = collection.get()

print("All data (default): \n")
results_default

All data (default): 



{'ids': ['doc1',
  'doc2',
  '63445d25-28d0-4813-b5de-b288988a1356',
  '3cbe0c7c-eb27-4050-b219-bfa47d1408e7'],
 'embeddings': None,
 'documents': ['Machine learning is amazing.',
  'Deep learning powers AI.',
  'AI is transforming industries.',
  'Vector databases are essential for RAG.'],
 'uris': None,
 'data': None,
 'metadatas': [{'topic': 'ml'}, {'topic': 'ai'}, None, None],
 'included': [<IncludeEnum.documents: 'documents'>,
  <IncludeEnum.metadatas: 'metadatas'>]}

In [10]:
# Fetch all (including embeddings)
results_with_embeddings = collection.get(include=['embeddings', 'documents', 'metadatas'])

print("All data (with embeddings): \n")
results_with_embeddings

All data (with embeddings): 



{'ids': ['doc1',
  'doc2',
  '63445d25-28d0-4813-b5de-b288988a1356',
  '3cbe0c7c-eb27-4050-b219-bfa47d1408e7'],
 'embeddings': array([[0.1       , 0.2       , 0.30000001],
        [0.40000001, 0.5       , 0.60000002],
        [0.1       , 0.2       , 0.30000001],
        [0.40000001, 0.5       , 0.60000002]]),
 'documents': ['Machine learning is amazing.',
  'Deep learning powers AI.',
  'AI is transforming industries.',
  'Vector databases are essential for RAG.'],
 'uris': None,
 'data': None,
 'metadatas': [{'topic': 'ml'}, {'topic': 'ai'}, None, None],
 'included': [<IncludeEnum.embeddings: 'embeddings'>,
  <IncludeEnum.documents: 'documents'>,
  <IncludeEnum.metadatas: 'metadatas'>]}

In [11]:
# Query similar documents using embedding
query_results = collection.query(
    query_embeddings =[[0.1, 0.2, 0.3]],
    n_results        =1,
    include          =["documents", "metadatas", "embeddings"]
)

print("Query results: \n")

query_results

Query results: 



{'ids': [['63445d25-28d0-4813-b5de-b288988a1356']],
 'embeddings': [array([[0.1       , 0.2       , 0.30000001]])],
 'documents': [['AI is transforming industries.']],
 'uris': None,
 'data': None,
 'metadatas': [[None]],
 'distances': None,
 'included': [<IncludeEnum.embeddings: 'embeddings'>,
  <IncludeEnum.documents: 'documents'>,
  <IncludeEnum.metadatas: 'metadatas'>]}

**UPDATE – Modify Data**

In [12]:
print(collection.get()['ids'])

['doc1', 'doc2', '63445d25-28d0-4813-b5de-b288988a1356', '3cbe0c7c-eb27-4050-b219-bfa47d1408e7']


In [13]:
# Update (upsert) document with the same ID
collection.upsert(
    ids       = ['para1'],
    documents = ["AgenticAI is majorly advanced S/W engg"],
    metadatas = [{"topic": "AgenticAI"}],
    embeddings= [[0.99, 0.99, 0.99]]
)

In [14]:
# Fetch updated doc
updated_doc = collection.get(ids=["doc1"], include=["documents", "metadatas", "embeddings"])
updated_doc

{'ids': ['doc1'],
 'embeddings': array([[0.1       , 0.2       , 0.30000001]]),
 'documents': ['Machine learning is amazing.'],
 'uris': None,
 'data': None,
 'metadatas': [{'topic': 'ml'}],
 'included': [<IncludeEnum.embeddings: 'embeddings'>,
  <IncludeEnum.documents: 'documents'>,
  <IncludeEnum.metadatas: 'metadatas'>]}

**adding data without embeddings**

In [15]:
try:
    collection.add(
        ids=["id1", "id2", "id3"],
        documents=["lorem ipsum", "doc2", "doc3"],
        metadatas=[
            {"chapter": 3, "verse": 16},
            {"chapter": 3, "verse": 5},
            {"chapter": 29, "verse": 11}
        ]
    )
except Exception as e:
    print(f"! Error while adding documents: {e}")

! Error while adding documents: Embedding dimension 384 does not match collection dimensionality 3


- It's trying to fallback to a default embedding function (OpenAI) → produces 384-d vectors.
- This does not match the collection’s expected 3 dimension.

- It uses Sentence Transformers → `all-MiniLM-L6-v2` model.
- Runs locally (downloads weights automatically on first use).
- Produces 384-dimensional embeddings for each document or text.

In [17]:
# Delete collection by name
chroma_client.delete_collection(name="my_collection")

print("Collection 'my_collection' deleted successfully.")

Collection 'my_collection' deleted successfully.


In [18]:
collection = chroma_client.create_collection(name="my_collection")

In [19]:
# Add data WITHOUT providing embeddings
collection.add(
    ids      =["id1", "id2"],
    documents=[
        "Artificial Intelligence is transforming industries.",
        "Vector databases are crucial for Retrieval Augmented Generation."
    ],
    metadatas=[{"topic": "AI"}, {"topic": "RAG"}]
)

print("Data added successfully (auto embeddings applied).\n")

Data added successfully (auto embeddings applied).



In [20]:
# Fetch data INCLUDING embeddings
results = collection.get(include=["embeddings", "documents", "metadatas"])

print("Data fetched from ChromaDB:")
print("IDs:", results["ids"])
print("Documents:", results["documents"])
print("Embedding vector size:", len(results["embeddings"][0]))  # Dimension size

Data fetched from ChromaDB:
IDs: ['id1', 'id2']
Documents: ['Artificial Intelligence is transforming industries.', 'Vector databases are crucial for Retrieval Augmented Generation.']
Embedding vector size: 384


**delete data**

In [21]:
collection.get(include=["documents"])  # ids are always included by default

{'ids': ['id1', 'id2'],
 'embeddings': None,
 'documents': ['Artificial Intelligence is transforming industries.',
  'Vector databases are crucial for Retrieval Augmented Generation.'],
 'uris': None,
 'data': None,
 'metadatas': None,
 'included': [<IncludeEnum.documents: 'documents'>]}

In [22]:
collection.delete(
    ids = ["id1", "id2", "id3"]
)

# delete() is silent on missing IDs → no error or warning.

Delete of nonexisting embedding ID: id3
Delete of nonexisting embedding ID: id3


#### Query and Get Data from Chroma Collections

In [23]:
# Add documents (some related to philosophy, some unrelated)
collection.add(
    ids=["doc1", "doc2", "doc3", "doc4", "doc5"],
    documents=[
        "Thus Spake Zarathustra is a philosophical novel by Friedrich Nietzsche.",
        "The Oracle speaks in riddles to guide the seekers of truth.",
        "Python is a popular programming language for machine learning.",
        "Quantum mechanics explores the nature of particles and waves.",
        "Philosophers debate about morality, free will, and the meaning of life."
    ],
    metadatas=[
        {"topic": "philosophy"},
        {"topic": "philosophy"},
        {"topic": "technology"},
        {"topic": "science"},
        {"topic": "philosophy"}
    ]
)

Chroma will use the collection's embedding function (default) to embed our text data

In [24]:
collection.query(
    query_texts=["thus spake zarathustra", "the oracle speaks"]
)

Number of requested results 10 is greater than number of elements in index 5, updating n_results = 5


{'ids': [['doc1', 'doc2', 'doc5', 'doc4', 'doc3'],
  ['doc2', 'doc3', 'doc5', 'doc1', 'doc4']],
 'embeddings': None,
 'documents': [['Thus Spake Zarathustra is a philosophical novel by Friedrich Nietzsche.',
   'The Oracle speaks in riddles to guide the seekers of truth.',
   'Philosophers debate about morality, free will, and the meaning of life.',
   'Quantum mechanics explores the nature of particles and waves.',
   'Python is a popular programming language for machine learning.'],
  ['The Oracle speaks in riddles to guide the seekers of truth.',
   'Python is a popular programming language for machine learning.',
   'Philosophers debate about morality, free will, and the meaning of life.',
   'Thus Spake Zarathustra is a philosophical novel by Friedrich Nietzsche.',
   'Quantum mechanics explores the nature of particles and waves.']],
 'uris': None,
 'data': None,
 'metadatas': [[{'topic': 'philosophy'},
   {'topic': 'philosophy'},
   {'topic': 'philosophy'},
   {'topic': 'science'

Chroma will use the collection's embedding function to embed your text queries, and use the output to run a vector similarity search against your collection.

In [25]:
# External embedding generation (for demonstration only)
# Using the same model as Chroma (Sentence Transformers) - default
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("all-MiniLM-L6-v2")

query_embeddings = model.encode([
    "thus spake zarathustra",
    "the oracle speaks"
]).tolist()   # Convert to list for Chroma

In [26]:
# Query using embeddings directly
results = collection.query(
    query_embeddings=query_embeddings,
    n_results       =3,
    include         =["documents"]
)

print("Query results: \n")
results

Query results: 



{'ids': [['doc1', 'doc2', 'doc5'], ['doc2', 'doc3', 'doc5']],
 'embeddings': None,
 'documents': [['Thus Spake Zarathustra is a philosophical novel by Friedrich Nietzsche.',
   'The Oracle speaks in riddles to guide the seekers of truth.',
   'Philosophers debate about morality, free will, and the meaning of life.'],
  ['The Oracle speaks in riddles to guide the seekers of truth.',
   'Python is a popular programming language for machine learning.',
   'Philosophers debate about morality, free will, and the meaning of life.']],
 'uris': None,
 'data': None,
 'metadatas': None,
 'distances': None,
 'included': [<IncludeEnum.documents: 'documents'>]}

#### where argument for metadata filtering

In [22]:
chroma_client.delete_collection(name="test_manual_filter")

ValueError: Collection test_manual_filter does not exist.

In [27]:
collection = chroma_client.create_collection(name="test_manual_filter")

In [28]:
# Add sample documents with metadata
collection.add(
    ids=["d1", "d2", "d3", "d4"],
    documents=[
        "This is a sample page about machine learning.",
        "This page describes vector databases and ChromaDB.",
        "A random text about cooking recipes and food.",
        "Another page about ChromaDB search functionality."
    ],
    embeddings=[
        [11.0, 12.0, 13.0],
        [1.2,  2.1,  3.4],
        [5.1,  6.2,  7.3],
        [1.0,  2.0,  3.0]
    ],
    metadatas=[
        {"page": 10},
        {"page": 10},
        {"page": 20},
        {"page": 10}
    ]
)


In [29]:
# Query with embeddings, metadata filter, and document filter
results = collection.query(
    query_embeddings=[
        [11.1, 12.1, 13.1],
        [1.1,  2.3,  3.2]
    ],  
    n_results     = 5,
    where         = {"page": 10},                     # Filter by metadata
    where_document= {"$contains": "ChromaDB"},        # Filter by document text
    include       = ["documents", "metadatas"]
)

print("Results:")
results

Number of requested results 5 is greater than number of elements in index 4, updating n_results = 4


Results:


{'ids': [['d2', 'd4'], ['d2', 'd4']],
 'embeddings': None,
 'documents': [['This page describes vector databases and ChromaDB.',
   'Another page about ChromaDB search functionality.'],
  ['This page describes vector databases and ChromaDB.',
   'Another page about ChromaDB search functionality.']],
 'uris': None,
 'data': None,
 'metadatas': [[{'page': 10}, {'page': 10}], [{'page': 10}, {'page': 10}]],
 'distances': None,
 'included': [<IncludeEnum.documents: 'documents'>,
  <IncludeEnum.metadatas: 'metadatas'>]}

Chroma processes each embedding vector as a separate query.

Since both embeddings are close to the same documents, you got the same top 2 matches for both queries.

If you want similarity scores and embeddings in results:

In [31]:
results = collection.query(
    query_embeddings=[
        [11.1, 12.1, 13.1],
        [1.1, 2.3, 3.2]
    ],
    n_results     = 5,
    where         = {"page": 10},
    where_document= {"$contains": "ChromaDB"},
    include       = ["documents", "metadatas", "embeddings", "distances"]
)

results

{'ids': [['d2', 'd4'], ['d2', 'd4']],
 'embeddings': [array([[1.20000005, 2.0999999 , 3.4000001 ],
         [1.        , 2.        , 3.        ]]),
  array([[1.20000005, 2.0999999 , 3.4000001 ],
         [1.        , 2.        , 3.        ]])],
 'documents': [['This page describes vector databases and ChromaDB.',
   'Another page about ChromaDB search functionality.'],
  ['This page describes vector databases and ChromaDB.',
   'Another page about ChromaDB search functionality.']],
 'uris': None,
 'included': ['documents', 'metadatas', 'embeddings', 'distances'],
 'data': None,
 'metadatas': [[{'page': 10}, {'page': 10}], [{'page': 10}, {'page': 10}]],
 'distances': [[292.10003662109375, 306.030029296875],
  [0.09000004082918167, 0.14000000059604645]]}

In [32]:
collection = chroma_client.create_collection(name="metadata_filter_example")

In [33]:
# Add data with metadata
collection.add(
    ids=["doc1", "doc2", "doc3", "doc4"],
    documents=[
        "This is page 10 about philosophy and life.",
        "This is page 20 about technology and AI.",
        "This is page 10 about machine learning basics.",
        "This is page 30 about cooking and recipes."
    ],
    metadatas=[
        {"page": 10},
        {"page": 20},
        {"page": 10},
        {"page": 30}
    ]
)

print("Data added successfully.\n")

Data added successfully.



In [36]:
# Query using metadata filter (page = 10)
results = collection.query(
    query_texts=["machine learning", "philosophy", "cricket"],
    where      ={"page": 10},
    n_results  =3,
    include    =["documents", "metadatas"]
)

In [37]:
# Display results
print("Filtered Query Results:")
for i, query in enumerate(["machine learning", "philosophy"]):
    print(f"\nQuery: {query}")
    for doc, meta in zip(results["documents"][i], results["metadatas"][i]):
        print(f" - {doc} (page: {meta['page']})")

Filtered Query Results:

Query: machine learning
 - This is page 10 about machine learning basics. (page: 10)
 - This is page 10 about philosophy and life. (page: 10)

Query: philosophy
 - This is page 10 about philosophy and life. (page: 10)
 - This is page 10 about machine learning basics. (page: 10)


- Metadata filter (`where`): restricts search to documents with specific metadata.
- Document text filter (`where_document`): restricts search to documents containing a given substring.
- For remaining candidates after filtering: Compute cosine similarity (default) or L2 distance between the query embedding and document embeddings.
- Rank and Select Top-N

#### Full Text Search and Regex

In [38]:
collection = chroma_client.create_collection(name="text_filter_demo")

In [39]:
# Add documents with some variations in text
collection.add(
    ids=["doc1", "doc2", "doc3", "doc4", "doc5"],
    documents=[
        "This is a guide about Machine Learning basics.",
        "Artificial Intelligence is transforming industries.",
        "Regex patterns can filter text effectively.",
        "Machine Learning and AI are related fields.",
        "Cooking recipes are unrelated to AI or ML."
    ],
    metadatas=[
        {"page": 1},
        {"page": 2},
        {"page": 3},
        {"page": 4},
        {"page": 5}
    ]
)

In [87]:
# Retrieve documents containing "Machine Learning"
results = collection.get(
    where_document={"$contains": "Machine Learning"},
    include=["documents"]
)

In [88]:
print("🔹 Contains 'Machine Learning':", results)

🔹 Contains 'Machine Learning': {'ids': [], 'embeddings': None, 'documents': [], 'uris': None, 'included': ['documents'], 'data': None, 'metadatas': None}


In [89]:
# Retrieve documents NOT containing "AI"
results_not = collection.get(
    where_document={"$not_contains": "AI"},
    include=["documents"]
)

print("🔹 NOT Contains 'AI':", results_not)

🔹 NOT Contains 'AI': {'ids': ['d1', 'd2', 'd3', 'd4'], 'embeddings': None, 'documents': ['This is a sample page about machine learning.', 'This page describes vector databases and ChromaDB.', 'A random text about cooking recipes and food.', 'Another page about ChromaDB search functionality.'], 'uris': None, 'included': ['documents'], 'data': None, 'metadatas': None}


Using \\$regex and \\$not_regex (Regex Pattern Matching)

In [90]:
# Regex search for documents that start with 'Artificial'
regex_results = collection.get(
    where_document={"$regex": "^Artificial"},
    include=["documents"]
)

print("🔹 Regex '^Artificial':", regex_results)

🔹 Regex '^Artificial': {'ids': [], 'embeddings': None, 'documents': [], 'uris': None, 'included': ['documents'], 'data': None, 'metadatas': None}


In [91]:
# Regex search for documents that do NOT contain 'Cooking'
not_regex_results = collection.get(
    where_document={"$not_regex": "Cooking"},
    include=["documents"]
)

print("🔹 NOT Regex 'Cooking':", not_regex_results)

🔹 NOT Regex 'Cooking': {'ids': ['d1', 'd2', 'd3', 'd4'], 'embeddings': None, 'documents': ['This is a sample page about machine learning.', 'This page describes vector databases and ChromaDB.', 'A random text about cooking recipes and food.', 'Another page about ChromaDB search functionality.'], 'uris': None, 'included': ['documents'], 'data': None, 'metadatas': None}
